In [ ]:
from pathlib import Path
from lxml import etree
import pandas as pd
import glob

# ==================================================
# RUTAS
# ==================================================

RUTA_BASE = Path(r"C:\Users\NI38504\Documents\Home\REMESAS\INSUMOS\Julio")
#RUTA_BASE = Path(r"\\172.25.16.15\Reportes Cobis_BI\Remesas\2026\Julio")

RUTA_SALIDA = Path(r"C:\Users\NI38504\Documents\Home\REMESAS\INSUMOS\Julio\CONSOLIDADO")

RUTA_SALIDA.mkdir(exist_ok=True)

ARCHIVO_SALIDA = (RUTA_SALIDA / "Consolidado_Remesas_di.xlsx")

# ==================================================
# NAMESPACE XML EXCEL
# ==================================================

ns = {"ss": "urn:schemas-microsoft-com:office:spreadsheet"}

# ==================================================
# ACUMULADOR
# ==================================================

df_total = pd.DataFrame()

# ==================================================
# RECORRER CARPETAS
# ==================================================

for carpeta in RUTA_BASE.iterdir():

    # Estos días tienen la estructura anterior, hay que obtenerlos manualmente
    black_list = [
        r"\\172.25.16.15\Reportes Cobis_BI\Remesas\2026\Julio\01072026",
        r"\\172.25.16.15\Reportes Cobis_BI\Remesas\2026\Julio\02072026",
        r"\\172.25.16.15\Reportes Cobis_BI\Remesas\2026\Julio\03072026",
        r"C:\Users\NI38504\Documents\Home\REMESAS\INSUMOS\Julio\01072026",
        r"C:\Users\NI38504\Documents\Home\REMESAS\INSUMOS\Julio\02072026",
        r"C:\Users\NI38504\Documents\Home\REMESAS\INSUMOS\Julio\03072026",
    ]

    if str(carpeta) in black_list:
        continue

    if not carpeta.is_dir():
        continue

    archivos = glob.glob(
        str(carpeta / "remesasdi*.xml")
    )

    if not archivos:
        continue

    for archivo in archivos:

        print(f"Procesando: {archivo}")

        try:

            tree = etree.parse(archivo)

            rows = []

            for row in tree.xpath(
                "//ss:Row",
                namespaces=ns
            ):

                valores = []

                for cell in row.xpath(
                    "./ss:Cell",
                    namespaces=ns
                ):

                    data = cell.xpath(
                        "./ss:Data/text()",
                        namespaces=ns
                    )

                    valores.append(
                        data[0].strip()
                        if data
                        else ""
                    )

                rows.append(valores)

            if not rows:
                continue

            # ==========================================
            # BUSCAR REMESAS PAGADAS
            # ==========================================

            indice_remesas = None

            for i, fila in enumerate(rows):

                texto = " ".join(
                    str(x).strip().upper() for x in fila if str(x).strip()
                )

                if "REMESAS PAGADAS" in texto:
                    indice_remesas = i
                    break

            if indice_remesas is None:

                print(f"No se encontró REMESAS PAGADAS en {archivo}")
                continue

            # ==========================================
            # ENCABEZADO
            # ==========================================

            encabezados = [
                str(x).strip() for x in rows[indice_remesas + 1]
            ]

            cantidad_columnas = len(encabezados)

            """ Obtengo el indice de la columna """
            # print(encabezados)

            # ==========================================
            # DATOS
            # ==========================================

            registros = []

            for fila in rows[indice_remesas + 2:]:

                # Temporal: necesito estos índices para que coincida con la estructura actual del proceso.
                mapeo_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 17, 18, 20, 21, 22, 23, 25, 26, 27, 28]
                
                #print(fila)
                fila = [fila[i] for i in mapeo_indices]
                #print(fila)

                # ignorar filas vacías
                if not any(str(x).strip() for x in fila):
                    continue

                # completar columnas faltantes
                if len(fila) < cantidad_columnas:
                    fila = fila + [""] * (
                        cantidad_columnas - len(fila)
                    )

                # recortar sobrantes
                if len(fila) > cantidad_columnas:
                    fila = fila[:cantidad_columnas]

                registros.append(fila)

            if not registros:
                continue

            # ==========================================
            # DF DEL ARCHIVO
            # ==========================================

            df = pd.DataFrame(
                registros,
                columns=encabezados
            )

            df["CARPETA_ORIGEN"] = carpeta.name

            # ==========================================
            # ACUMULAR
            # ==========================================

            df_total = pd.concat(
                [df_total, df],
                ignore_index=True
            )

        except Exception as e:

            print(f"Error procesando {archivo}")
            print(str(e))

# ==================================================
# EXPORTAR
# ==================================================

if not df_total.empty:

    with pd.ExcelWriter(
        ARCHIVO_SALIDA,
        engine="openpyxl"
    ) as writer:

        df_total.to_excel(
            writer,
            sheet_name="REMESAS PAGADAS",
            index=False
        )

    print("\nProceso finalizado.")
    print(f"Archivo generado: {ARCHIVO_SALIDA}")
    print(f"Total registros: {len(df_total):,}")

else:

    print("No se encontrron registros para consolidar.")

Procesando: C:\Users\NI38504\Documents\Home\REMESAS\INSUMOS\Julio\04072026\remesasdi_RIA_0404072026.xml
['12542566809', 'RIA', 'FULBIA VERONICA DAVILA SILVA', 'DORCA ABIGAIL DAVILA SILVA', 'CEDULA', '0010410021036G', '04/07/2026', '04/07/2026', 'MANAGUA', 'USD', '280.00', '280.00', '0.00', '1.00', '0.00', '0.00', '0.00', '280.00', '0.00', '0.00', '172', 'SUCURSAL PLAZA ESPAÃ\x91A', 'Efectivo', '', '', 'AHORRO PERSONAL', '1275245204', 'ni38270', 'Alvaro Jose Tapia Perez', 'ATX']
['12542566809', 'RIA', 'FULBIA VERONICA DAVILA SILVA', 'DORCA ABIGAIL DAVILA SILVA', 'CEDULA', '0010410021036G', '04/07/2026', '04/07/2026', 'MANAGUA', 'USD', '280.00', '280.00', '1.00', '280.00', '0.00', '172', 'SUCURSAL PLAZA ESPAÃ\x91A', 'Efectivo', '', 'AHORRO PERSONAL', '1275245204', 'ni38270', 'Alvaro Jose Tapia Perez']
Procesando: C:\Users\NI38504\Documents\Home\REMESAS\INSUMOS\Julio\04072026\remesasdi_RMI_0404072026.xml
Procesando: C:\Users\NI38504\Documents\Home\REMESAS\INSUMOS\Julio\04072026\remesasdi_